# DentVLM panoramic pipeline (Kaggle runner)

One image goes through the paper's own protocol; every question is one DentVLM was trained on:

```text
panoramic X-ray
   -> 13 yes/no questions, one per DentVLM panoramic task (Supplementary Table 7 wording), whole image
   -> line 1 of each reply = Yes/No; the rationale names the location with the nine trained
      descriptors -> six dental-arch cells; multiplicity = number of distinct cells the finding is
      reported in (never a tooth count), with a status: resolved, partial, not stated, unresolved
   -> optional knobs: phrasings=3 vote, location="regions" (every cell asked every task, whatever the
      whole image answered; the whole-image answers are kept separately)
   -> JSON per image, deterministic dentist summary, TP/FP/TN/FN evaluation per finding and per cell,
      and occupied-region counts (predicted cells vs. the distinct cells the true boxes occupy; the
      "counting" knob, independent of "evaluate_location")
   -> location truth: ground-truth boxes are translated into the same six cells by a vision LLM
      (numbered boxes drawn on the image -> FDI quadrant x anterior/posterior), by DentVLM itself
      (experimental), or by fixed windows
   -> dentist report: a text LLM (the reporter role) gets the findings of one image as one dense JSON (every
      finding and extra task with its question and answer, every region, multiplicity, explicit statuses) and
      returns a classified report in the dentist's language (report_language); verified against the data,
      corrected once if needed, rendered to Markdown
   -> reading all of that text: strict readers (a regex for Yes/No, the nine location descriptors, JSON
      loaders) by default, and optionally a parser LLM (the parser role) for the replies they cannot read
      or would read too literally - per stage, or with one global "parser_mode" switch
```

**CELL 3 is the only cell to edit.** It holds one dictionary per configuration: a name plus the knobs that
configuration changes - local or hosted backend, the analyzer / adapter / reporter models, the protocol knobs,
the location truth. Everything a configuration does not mention comes from `experiments.DEFAULTS`. Every
experiment answers the same images, writes into `<output_root>/<name>/`, and CELL 11 scores them side by side
and ranks them, so one session says which configuration works best instead of one configuration per session.
Run the cells in order.

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the project .py files
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py",
                          "llm_api.py", "llm_parser.py", "report_writer.py", "dental_analysis.py",
                          "experiments.py", "run_monitor.py", "response_cache.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import dental_analysis as da
import location_adapter as la
import llm_api
import llm_parser as lp
import run_monitor as mon
import experiments as xp
import report_writer as rw
from llama_runtime import LlamaCppServer, build_llama_cpp, convert_to_gguf, download_gguf, local_gguf
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# ============================================================
# CELL 3 - CONFIGURATION: the experiments to run and compare (the only cell to edit)
# ============================================================
# How much a run prints. Failures always print completely, whatever this says: the error, its traceback,
# and the full prompt and reply behind a parse failure. "auto" keeps one dense line per image and prints a
# single model call only when it is worth reading (slow, truncated, empty); "each" prints every call, for
# debugging one image; "off" prints none.
CALL_LOG = "auto"
LEDGER = mon.Ledger("session")   # every failure of every cell below, kept for the summary at the end

OUTPUT_ROOT = "/kaggle/working/dentvlm_sweep"

# The analyzer is always local DentVLM. The only hosted model in this notebook is the location adapter of
# CELL 10, which translates the ground-truth boxes into DentVLM's six regions (one call per image, once per
# dataset, shared by every experiment). Keys come from environment variables or Kaggle Secrets
# (Add-ons > Secrets); with no key at all, set LOCATION_TRUTH = "geometry" below and no hosted model is used.
PROVIDERS = {
    "openai": {"base_url": "https://api.openai.com/v1",
               "api_key": llm_api.secret("OPENAI_API_KEY", required=False)},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",
                   "api_key": llm_api.secret("OPENROUTER_API_KEY", required=False)},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "api_key": llm_api.secret("GEMINI_API_KEY", required=False)},
}
llm_api.configure_providers(PROVIDERS)
mon.CALL_LOG = CALL_LOG

# ------------------------------------------------------------------------------------------------------
# THE ONE DIAL: how many images of each dataset. Start small, read the leaderboard, raise it and rerun -
# every cell resumes, so nothing already answered is paid for twice. The budget lines printed at the bottom
# of this cell say what the current setting costs before a single call is made.
# ------------------------------------------------------------------------------------------------------
LIMIT = 20          # first N images per dataset (sorted by id); None = all (100 umfih_test + 50 dentex_val)

# How the ground-truth boxes reach DentVLM's six regions before location is scored.
#   "areas"    a hosted vision model marks where THIS patient's six regions lie, Python places every box
#   "llm"      a hosted vision model names the dental-arch unit of each numbered box drawn on the image
#   "geometry" fixed image windows, no hosted model, no key needed - but the midline, the canine line and
#              the occlusal plane move with patient positioning, so location scored this way is coarse
# CELL 10 prints how often each method matches the exact FDI cell on DENTEX ("adapter vs FDI truth"), next
# to the fixed windows, so this choice is measured rather than assumed.
LOCATION_TRUTH = "areas"
ADAPTER = {"provider": "openai", "model": "gpt-5", "token_param": "max_completion_tokens", "temperature": None,
           "max_output_tokens": 8192, "max_boxes_per_call": 12}
# With only an OpenRouter key instead:
# ADAPTER = {"provider": "openrouter", "model": "qwen/qwen3-vl-235b-a22b-instruct", "temperature": 0.0,
#            "max_output_tokens": 4096, "max_boxes_per_call": 12}

# What every experiment starts from. Everything not named here comes from experiments.DEFAULTS, which holds
# the authors' own inference settings (temperature 0, max_tokens 512, ctx 16384) - print(xp.DEFAULTS) for
# the full list with its comments.
SHARED = {
    "output_root": OUTPUT_ROOT,
    "backend": "local",           # DentVLM through llama.cpp: the only analyzer in this notebook

    # The paper's protocol. Each experiment below changes exactly one of these.
    "phrasings": 1,               # 1 wording per task; 3 in the phrasing experiment (2 could only tie)
    "region_vote": "union",       # only acts with phrasings > 1; CELL 11 replays "majority" for free
    "location": "rationale",      # regions read from the rationale, as the authors' scorer does
    "ask_untrained": False,       # True in the zero-shot experiment
    "extra_tasks": False,         # residual crown / eruption space / calculus have no UMFIH class, so they
                                  # would add 3 calls per image that no table scores. On only for a report.
    "parse_retries": 1,           # one format reminder after an unreadable Yes/No line; free when nothing fails
    "image_max_tokens": 8192,     # the authors' max_pixels; 1369 is their 1024x1024 ablation (see the note
                                  # under the experiment list)

    # Evaluation and report settings: no inference at all, so a saved run is re-scored by rerunning
    # CELLs 3, 10 and 11 with these changed.
    "evaluate_location": True,
    "counting": True,
    "location_truth": LOCATION_TRUTH,
    "adapter": ADAPTER,
    "location_failure_policy": "exclude",  # a box the adapter cannot place is excluded and counted as such,
                                           # never scored against the fixed windows next to adapted boxes
    "parser_mode": "code",        # the paper's strict readers (line-1 Yes/No, the nine descriptors)
    "report_images": 5,           # CELL 14 writes 5 dentist reports with the hosted "reporter" role; the
                                  # report is never scored, so this is a demo, not an experiment
}

# ------------------------------------------------------------------------------------------------------
# THE EXPERIMENTS. One knob each against the paper protocol, ordered by cost. Delete a line to skip it.
# "reads:" names the CELL 11/12 table that answers its question.
# ------------------------------------------------------------------------------------------------------
EXPERIMENTS = xp.build([
    # The paper's protocol: one whole-image yes/no question per task, regions read from the rationale.
    # The reference every other row is paired against.                                    10 calls/image
    {"name": "01-paper"},

    # Can DentVLM say anything about the five UMFIH classes it has no task for (furcation, apical surgery,
    # root resorption, orthodontic appliance, surgical plates)? This is the only way to cover the last 5 of
    # the 14 classes; the paper reports 52-64% on diseases it never saw.                  +5 calls/image
    # reads: the per-finding table, rows with trained_task=False (sensitivity against false alarms).
    {"name": "02-untrained", "ask_untrained": True},

    # Ask each task with three of the model's own wordings and vote. Is one answer stable, and does the vote
    # correct more than it breaks?                                                        +20 calls/image
    # reads: phrasing_votes (agree / disagree / tie per task), stage_changes first_phrasing_to_vote,
    # vote_replay_comparison (union vs majority, computed from the same answers at no extra cost).
    {"name": "03-phrasings3", "phrasings": 3},

    # The main workflow question: does asking each task once per dental-arch region - in the model's own
    # descriptor words, whole image every time - find what the whole-image question misses, and what does it
    # cost in false alarms? This is also the only mode where each region has its own answer.
    #                                                                                     +60 calls/image
    # reads: whole_image.csv next to presence.csv (findings recovered against false alarms added, per
    # finding), stage_changes whole_image_to_regions, region_presence.csv (each region judged on its own).
    {"name": "04-regions", "location": "regions"},
], shared=SHARED)
xp.show(EXPERIMENTS, parsers=False)

# Not in the list, on purpose: phrasings=2 (a 1-1 split is recorded as unresolved), location="none" (same
# calls, the rationale's regions thrown away), region_vote (CELL 11 replays it), a hosted analyzer (this
# notebook tests DentVLM), the parser LLM (it changes the reader, not the workflow). The authors' 1024x1024
# image bound is a one-line experiment - {"name": "05-image1369", "image_max_tokens": 1369} - worth adding
# only after the four above are read: it costs a full 10 calls/image because a changed image budget is a
# different reply, and it answers a runtime question rather than a workflow one.

# Datasets: the same images for every experiment, so CELL 11 compares them paired. umfih_test annotates all
# 14 classes; dentex_val carries FDI quadrant and tooth numbers, which give the exact region of every true
# box for caries, periapical lesion and impacted tooth - the only place location can be scored against
# certain truth. DentVLM's authors used only the official DENTEX test split, so this one is held out.
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": LIMIT,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    {"name": "dentex_val", "kind": "dentex", "limit": LIMIT,
     "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
     "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
    # Confirmation set for the winning configuration, once the sweep above has decided one:
    # {"name": "umfih_external", "kind": "yolo", "limit": LIMIT,
    #  "images": "/kaggle/working/umfih_14class/external/images",
    #  "labels": "/kaggle/working/umfih_14class/external/labels"},
]


def new_questions_per_image(configs):
    """How many questions each experiment adds that no earlier one in the list already asks.

    Identical local requests are served from <output_root>/_response_cache, so the base run is also
    phrasing 1 of the phrasing run and the whole-image stage of the region run. The cache key includes the
    server settings, so a changed image budget or context is a different reply and shares nothing.
    """
    seen, rows = set(), []
    for cfg in configs:
        protocol = xp.protocol(cfg)
        asked = {(xp.server_key(cfg), q) for t in protocol.tasks()
                 for q in dp.questions_for(t)[:protocol.phrasings]}
        if protocol.location == "regions":
            asked |= {(xp.server_key(cfg), dp.region_question(t, c)) for t in protocol.tasks() for c in dp.CELLS}
        rows.append((cfg["name"], len(asked), len(asked - seen)))
        seen |= asked
    return rows


images = (LIMIT * len(DATASETS)) if LIMIT else None
print(f"\nbudget  (LIMIT={LIMIT} -> {images if images else 'all ~150'} images)")
total = 0
for name, asked, fresh in new_questions_per_image(EXPERIMENTS):
    total += fresh
    print(f"  {name:<16} {asked:>3} questions/image, {fresh:>3} new to the GPU")
calls = total * (images or 150)
print(f"  {'TOTAL':<16} {total:>3} new calls/image = {calls:,} calls"
      f" ~ {calls * 3 / 3600:.1f} h at 3 s/call, {calls * 5 / 3600:.1f} h at 5 s/call")
print("  The first [DONE] line gives the real seconds per call on this GPU. Everything resumes, so a run"
      " that does not finish is continued by rerunning CELL 9, and LIMIT can be raised afterwards.")

# Machine settings for the local backend: where llama.cpp is built and served, and where the GGUF files are
# converted and cached. The same for every experiment; what the model is and sees (checkpoint, context,
# image tokens) is a knob of the experiment instead.
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentvlm"
SERVER_LOG_PATH = "/kaggle/working/llama_dentvlm_server.log"
SERVER_STARTUP_TIMEOUT = 300.0
MODEL_DIR = "/kaggle/working/models/dentvlm"
CONVERT_WORK_DIR = "/tmp/dentvlm_hf"  # scratch for the 17 GB safetensors; not persisted
HF_TOKEN_SECRET = "HF_TOKEN"          # Kaggle secret holding a Hugging Face token (or set env HF_TOKEN)
CUDA_ARCH = None                      # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4

LOCAL = [c for c in EXPERIMENTS if xp.is_local(c)]
print(f"\ndatasets = {[d['name'] for d in DATASETS]} | local experiments = {[c['name'] for c in LOCAL]}")

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if LOCAL:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("No local experiment; llama.cpp checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if LOCAL:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("No local experiment; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - Local DentVLM: the GGUF files, and one llama.cpp server at a time
# ============================================================
# DentVLM (Hugging Face ZJU-AI4H/DentVLM, gated with automatic approval, CC BY-NC 4.0) ships as bf16
# safetensors; model_source "convert" downloads and converts it once (~17 GB scratch, ~9.5 GB kept),
# "local" uses files already in MODEL_DIR (an attached Kaggle dataset), "hf" downloads them from your own
# gguf_repo_id. Experiments asking for the same files convert or download once; experiments asking for the
# same server settings share the running server.
import requests

MODELS = {}
for cfg in LOCAL:
    key = xp.model_key(cfg)
    if key in MODELS:
        continue
    source, repo_id, model_filename, mmproj_filename = key
    hf_token = llm_api.secret(HF_TOKEN_SECRET, required=False)
    # A conversion or download that fails takes only the experiments needing these files; the error is in
    # the ledger and the sweep reports them as failed instead of dying here.
    with mon.guard(f"model files {model_filename}", LEDGER) as step:
        if source == "convert":
            if not hf_token:
                print(f"No Hugging Face token under {HF_TOKEN_SECRET!r}; only model_source 'local' works without one.")
            MODELS[key] = convert_to_gguf(MODEL_DIR, LLAMA_CPP_DIR, hf_token=hf_token, work_dir=CONVERT_WORK_DIR,
                                          model_filename=model_filename, mmproj_filename=mmproj_filename)
            print("Keep the two files above (private Kaggle dataset or Hugging Face repo) and switch "
                  "model_source to 'local' or 'hf' for the next session.")
        elif source == "local":
            MODELS[key] = local_gguf(MODEL_DIR, model_filename, mmproj_filename)
        else:
            MODELS[key] = download_gguf(MODEL_DIR, repo_id, model_filename, mmproj_filename, hf_token)
    if not step.ok:
        continue
    for label, path in (("Language model", MODELS[key].model_path), ("Vision projector", MODELS[key].mmproj_path)):
        print(f"{label}: {path} ({Path(path).stat().st_size / 1024**3:.2f} GiB)")

SERVER = SERVER_SETTINGS = None


def local_server(cfg):
    """The llama.cpp server for one experiment, restarted only when its local settings change."""
    global SERVER, SERVER_SETTINGS
    if SERVER is not None and SERVER_SETTINGS == xp.server_key(cfg):
        return SERVER
    if SERVER is not None:
        SERVER.stop()
    files = MODELS[xp.model_key(cfg)]
    SERVER = LlamaCppServer(binary=LLAMA_SERVER, model_path=files.model_path, mmproj_path=files.mmproj_path,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS,
                            n_gpu_layers=cfg["n_gpu_layers"], ctx_size=cfg["ctx_size"],
                            image_max_tokens=cfg["image_max_tokens"], image_min_tokens=cfg["image_min_tokens"],
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    SERVER.start(reuse_existing=False)
    ids = [m.get("id") for m in requests.get(f"{SERVER.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    SERVER_SETTINGS = xp.server_key(cfg)
    print(f"{cfg['name']}: DentVLM server verified at {SERVER.base_url}")
    return SERVER


def open_runner(cfg):
    """The analyzer of one experiment: its hosted model, or the local server started above."""
    return xp.runner(cfg, local_server(cfg) if xp.is_local(cfg) else None)


# One reader per experiment, built once and shared by its analyzer run, its location adapter and its
# report writer, so every parser call of that experiment is counted once and one fingerprint says how
# the whole experiment read its replies.
PARSERS = {}


def open_parser(cfg):
    if cfg["name"] not in PARSERS:
        PARSERS[cfg["name"]] = xp.parser(cfg)
    return PARSERS[cfg["name"]]


def read_reply(parser, question, reply):
    """(answer, regions, unresolved) of one raw reply, read exactly as the run will read it.

    Used by the smoke test and the free-form cell so what they print is what the configured readers
    produce, not a second opinion from the strict readers alone.
    """
    if parser is None or not parser.policy.uses_llm():
        return dp.extract_answer(reply["text"]), dp.extract_regions(reply["text"]), False
    decision = parser.decision("whole_image_decision", reply["text"], question,
                               truncated=bool(reply.get("truncated")))
    if decision.value != "yes":
        return decision.value, [], not decision.resolved
    located = parser.location("rationale_location", reply["text"], question=question,
                              truncated=bool(reply.get("truncated")))
    return decision.value, list(located.value or []), not located.resolved


print("Model files ready:", len(MODELS), "| local server and parser helpers defined")

In [ ]:
# ============================================================
# CELL 7 - Load ground truth for every dataset (no model calls)
# ============================================================
# Each dataset is loaded behind a guard: a missing folder or a malformed label file prints in full and the
# other datasets still load, and a dataset that fails is dropped from DATASETS so the cells below stay
# consistent. truth_report prints what was loaded and anything unusable in it (images with no label file,
# labels with no image, annotated images missing from disk) before a single model call is paid for.
GT = {}
for spec in DATASETS:
    with mon.guard(f"load {spec['name']}", LEDGER) as step:
        if spec["kind"] == "yolo":
            gt = ev.load_yolo(spec["images"], spec["labels"])
        elif spec["kind"] == "dentex":
            gt = ev.load_dentex(spec["images"], spec["annotations"])
        else:
            raise ValueError(f"unknown dataset kind {spec['kind']!r}")
        if spec.get("limit"):
            gt = dict(sorted(gt.items())[: spec["limit"]])
        GT[spec["name"]] = gt
        ev.truth_report(gt, spec["name"])
DATASETS = [d for d in DATASETS if d["name"] in GT]
if not DATASETS:
    raise RuntimeError("no dataset loaded; fix the paths reported above and rerun this cell")
print("datasets ready:", [d["name"] for d in DATASETS])

In [ ]:
# ============================================================
# CELL 8 - Smoke test: raw replies of every experiment on a few images
# ============================================================
# Sends the primary implant and caries questions to the first images ("smoke_images" in CELL 3; 0 skips it).
# Expect line 1 to be Yes/No on every reply, a location descriptor on most Yes replies, and no truncation.
# The answers are read with this experiment's own readers (CELL 3 "parser_mode"), so what is printed here
# is what the run will record - not a second opinion from the strict readers.
SMOKE = {}
for cfg in [c for c in EXPERIMENTS if c["smoke_images"]]:
    with mon.guard(f"smoke {cfg['name']}", LEDGER) as step:
        runner, parser = open_runner(cfg), open_parser(cfg)
        paths = [g["path"] for _, g in sorted(next(iter(GT.values())).items())[: cfg["smoke_images"]]]
        rows = []
        for path in paths:
            for task in ("implant", "caries"):
                question = dp.questions_for(task)[0]
                reply = runner.ask(path, question)
                answer, regions, unresolved = read_reply(parser, question, reply)
                rows.append({"experiment": cfg["name"], "image": Path(path).name, "task": task,
                             "answer": answer, "regions": regions, "location_unresolved": unresolved,
                             "truncated": reply["truncated"], "text": reply["text"]})
                print(f"--- {cfg['name']} | {rows[-1]['image']} | {task} | answer={answer} | "
                      f"regions={regions}{' (unresolved)' if unresolved else ''} | "
                      f"truncated={reply['truncated']}\n{reply['text'][:600]}\n")
        SMOKE[cfg["name"]] = rows
        yes_rows = [r for r in rows if r["answer"] == "yes"]
        print(f"{cfg['name']}: parsed answers {sum(r['answer'] is not None for r in rows)}/{len(rows)} | "
              f"yes replies naming a region {sum(bool(r['regions']) for r in yes_rows)}/{len(yes_rows)} | "
              f"unresolved locations {sum(r['location_unresolved'] for r in rows)} | "
              f"truncated {sum(r['truncated'] for r in rows)}\n")
        if parser.policy.uses_llm():
            print(f"{cfg['name']}: parser usage {parser.usage}\n")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_ROOT, "smoke.json").write_text(json.dumps(SMOKE, indent=1, ensure_ascii=False), encoding="utf-8")

In [ ]:
# ============================================================
# CELL 8.5 - Free-form image + prompt inference (local DentVLM or API)
# ============================================================
# Edit only these four values. The selected experiment decides whether the request uses the local
# DentVLM server or a hosted API model. Set CUSTOM_IMAGE_PATH to any X-ray path; None uses the
# first image from the first dataset. Add optional look-alike prompts to CUSTOM_PROMPT_VARIANTS.
RUN_CUSTOM_INFERENCE = False
CUSTOM_EXPERIMENT = EXPERIMENTS[0]["name"]  # any name from CELL 3
CUSTOM_IMAGE_PATH = None  # e.g. "/kaggle/input/my-dataset/panoramic.png"
CUSTOM_PROMPT = "Based on the imaging, determine whether the patient has caries?"
CUSTOM_PROMPT_VARIANTS = [
    # "Can you identify any carious lesions in this panoramic radiograph?",
    # "Does this X-ray show caries? If yes, explain where it is visible.",
]

CUSTOM_INFERENCE_RESULTS = []
CUSTOM_INFERENCE_RESULT = None
if RUN_CUSTOM_INFERENCE:
    matches = [cfg for cfg in EXPERIMENTS if cfg["name"] == CUSTOM_EXPERIMENT]
    if not matches:
        raise ValueError(f"Unknown CUSTOM_EXPERIMENT {CUSTOM_EXPERIMENT!r}; choose one of {[c['name'] for c in EXPERIMENTS]}")
    cfg = matches[0]
    if CUSTOM_IMAGE_PATH is None:
        _, first_ground_truth = next(iter(GT.items()))
        image_path = Path(next(iter(sorted(first_ground_truth.items())))[1]["path"])
    else:
        image_path = Path(CUSTOM_IMAGE_PATH)
    if not image_path.is_file():
        raise FileNotFoundError(f"X-ray image not found: {image_path}")

    prompts = [CUSTOM_PROMPT, *CUSTOM_PROMPT_VARIANTS]
    if not prompts or any(not isinstance(prompt, str) or not prompt.strip() for prompt in prompts):
        raise ValueError("CUSTOM_PROMPT and every CUSTOM_PROMPT_VARIANTS item must be a non-empty string")

    runner, parser = open_runner(cfg), open_parser(cfg)
    print(f"experiment={cfg['name']} | backend={cfg['backend']} | model={runner.model} | image={image_path}")
    print("  readers:", " ".join(f"{k}={v}" for k, v in parser.policy.resolved().items()))
    for index, prompt in enumerate(prompts, 1):
        reply = runner.ask(image_path, prompt)
        answer, regions, unresolved = read_reply(parser, prompt, reply)
        result = {
            "experiment": cfg["name"], "backend": cfg["backend"], "model": runner.model,
            "image": str(image_path), "prompt": prompt, "raw_reply": reply["text"],
            "parsed_yes_no": answer, "parsed_regions": regions,
            "location_unresolved": unresolved, "response": reply,
        }
        CUSTOM_INFERENCE_RESULTS.append(result)
        print(f"\n--- prompt {index}/{len(prompts)} ---\n{prompt}")
        print(f"--- raw reply | parsed_yes_no={answer} | parsed_regions={regions}"
              f"{' (unresolved)' if unresolved else ''} ---")
        print(reply["text"])
    CUSTOM_INFERENCE_RESULT = CUSTOM_INFERENCE_RESULTS[-1]
else:
    print("Custom inference is off. Edit the prompt/image above and set RUN_CUSTOM_INFERENCE = True.")

In [ ]:
# ============================================================
# CELL 9 - Run every experiment (resumable: finished images are skipped)
# ============================================================
# One experiment at a time, one directory each: <output_root>/<name>/<dataset>/. The resolved configuration
# is saved as experiment.json and hashed into the run manifest, so a changed knob can never be mixed into a
# resumed run. A failure prints in full, is recorded in LEDGER and stepped over: one bad image never costs
# the rest of the dataset (a rerun resumes it) and one bad experiment never costs the sweep. Three failed
# images in a row stop that dataset instead, because that is a dead server rather than a bad image.

FAILED = {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    print(f"\n{'=' * 78}\n=== {name}: {xp.analyzer_name(cfg)} | {xp.protocol(cfg)}\n{'=' * 78}")
    with mon.guard(f"run {name}", LEDGER) as step:
        xp.record(cfg)
        runner, parser = open_runner(cfg), open_parser(cfg)
        print(f"  runner = {runner.settings()}")
        print("\n".join("  " + line for line in parser.summary_lines()))
        for spec in DATASETS:
            images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
            out = dp.run_dataset(runner, images, xp.run_dir(cfg, spec["name"]), protocol=xp.protocol(cfg),
                                 resume=True, ledger=LEDGER, parser=parser,
                                 provenance=xp.provenance(cfg, llama_cpp_ref=LLAMA_CPP_REF))
            print("  saved:", out)
    if not step.ok:
        FAILED[name] = step.error

print("\nfinished:", [c["name"] for c in EXPERIMENTS if c["name"] not in FAILED], "| failed:", sorted(FAILED))
LEDGER.report(path=Path(OUTPUT_ROOT) / "failures.json")

In [ ]:
# ============================================================
# CELL 10 - Location truth: translate ground-truth boxes into DentVLM's six cells (resumable)
# ============================================================
# Independent of the model run, and keyed by the adapter rather than by the experiment: every experiment
# using the same adapter reads the same translated boxes (whether they serve the location tables, the
# occupied-region counts, or both: the count target is one region per box, so the same truth) from
# <output_root>/location_truth/<dataset>/<adapter>/ instead of paying for them again.
# One JSON per image under boxes/, drawn images under drawn/ for audit; unparseable replies follow
# location_failure_policy, and every attempt and fallback is saved.
ADAPTED, TRUTH_DIRS = {}, {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    for spec in DATASETS:
        dataset = spec["name"]
        if cfg["location_truth"] == "geometry":
            print(f"{name}/{dataset}: fixed windows (and FDI tooth numbers where the dataset has them)")
            continue
        if not xp.uses_location_truth(cfg):
            print(f"{name}/{dataset}: neither location nor counts are scored; no true box is placed")
            continue
        serves = ("location and counts" if cfg["evaluate_location"] and cfg["counting"]
                  else "counts only (location scoring is off)" if cfg["counting"] else "location only")
        out = xp.truth_dir(cfg, dataset)
        if out in TRUTH_DIRS:
            ADAPTED[name, dataset] = TRUTH_DIRS[out]
            print(f"{name}/{dataset}: reuses {out} ({serves})")
            continue
        print(f"{name}/{dataset}: adapted truth serves {serves}")
        # A failed adapter leaves this pair out of ADAPTED, and the evaluation then scores it against the
        # fixed windows; the failure stays in the ledger so the fallback is never silent.
        with mon.guard(f"location {name}/{dataset}", LEDGER, note="scoring falls back to the fixed windows") as step:
            adapter = xp.location_adapter(cfg, open_runner(cfg) if cfg["location_truth"] == "fdm" else None,
                                          parser=open_parser(cfg))
            TRUTH_DIRS[out] = ADAPTED[name, dataset] = la.adapt_dataset(adapter, GT[dataset], out, resume=True,
                                                                        ledger=LEDGER)
            print(f"{name}/{dataset}: {la.summarize(ADAPTED[name, dataset])}")
            agreement = ev.truth_agreement(GT[dataset], ADAPTED[name, dataset])
        if step.ok and agreement["boxes_with_fdi"]:
            # DENTEX carries FDI tooth numbers: exact cells, so this is the adapter's own accuracy.
            print(f"{name}/{dataset}: adapter vs FDI truth {agreement}")

In [ ]:
# ============================================================
# CELL 11 - Evaluate and compare every experiment
# ============================================================
import pandas as pd
from IPython.display import display

# Each experiment is scored against its own location truth and written to <name>/<dataset>/evaluation/, with
# its own evaluate_location and counting switches.
REPORTS = {}
for cfg in EXPERIMENTS:
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        # Scoring one experiment reads its saved results: a corrupt artifact or a dataset whose adapted
        # truth does not cover every image is reported and skipped, and the other experiments still rank.
        with mon.guard(f"score {name}/{dataset}", LEDGER) as step:
            results = dp.load_results(xp.run_dir(cfg, dataset))
            if not (set(GT[dataset]) & set(results)):
                LEDGER.note(f"score {name}/{dataset}", "no results yet; run CELL 9",
                            path=str(xp.run_dir(cfg, dataset)))
                continue
            truth = ev.apply_adapted(GT[dataset], ADAPTED[name, dataset]) if (name, dataset) in ADAPTED else GT[dataset]
            REPORTS[name, dataset] = ev.evaluate(truth, results, dataset=dataset,
                                                 out_dir=xp.run_dir(cfg, dataset) / "evaluation",
                                                 evaluate_location=cfg["evaluate_location"], counting=cfg["counting"])

# One joined artifact supplies the notebook and reusable CSVs. Full rows retain image IDs; the notebook
# shows compact decision tables so raw confusion counts, denominators, and protocol changes stay together.
VIEWS = da.compact_views(GT, REPORTS)
LEADERBOARD = pd.DataFrame(VIEWS["experiment_overview"])
FINDING_COMPARISON = pd.DataFrame(VIEWS["finding_comparison"])
SITUATION_COMPARISON = pd.DataFrame(VIEWS["situation_comparison"])
SITUATION_FINDING_COMPARISON = pd.DataFrame(VIEWS["situation_finding_comparison"])
STAGE_COMPARISON = pd.DataFrame(VIEWS["stage_comparison"])
PHRASING_COMPARISON = pd.DataFrame(VIEWS["phrasing_comparison"])
VOTE_REPLAY_COMPARISON = pd.DataFrame(VIEWS["vote_replay_comparison"])
PARSE_RECOVERY_COMPARISON = pd.DataFrame(VIEWS["parse_recovery_comparison"])
CALL_USAGE_COMPARISON = pd.DataFrame(VIEWS["call_usage_comparison"])
PARSER_USAGE_COMPARISON = pd.DataFrame(VIEWS["parser_usage_comparison"])

if not LEADERBOARD.empty:
    Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
    LEADERBOARD = LEADERBOARD.sort_values(["dataset", "f1"], ascending=[True, False])
    LEADERBOARD.to_csv(Path(OUTPUT_ROOT) / "leaderboard.csv", index=False)
    ev.write_report(VIEWS, Path(OUTPUT_ROOT) / "overview")
    protocol = ["dataset", "experiment", "phrasings", "region_vote", "location",
                "ask_untrained", "evaluate_location", "counting"]
    print("===== finding detection (rates use scored checks only) =====")
    display(LEADERBOARD[protocol + ["images", "annotated_checks", "not_assessed_checks",
        "expected_checks", "scored_checks", "TP", "TN", "FP", "FN",
        "excluded_unparseable_checks", "coverage", "sensitivity", "specificity", "ppv", "f1",
        "macro_f1", "mean_false_alarms_per_image", "logical_calls", "inference_calls", "cache_hits",
        "mean_calls_per_image", "mean_inference_calls_per_image", "cache_hit_rate"]].set_index(protocol))

    location_rows = LEADERBOARD[(LEADERBOARD["evaluate_location"] == True) &
                                (LEADERBOARD["location"] != "none")]
    if not location_rows.empty:
        print("===== DentVLM six-cell location =====")
        display(location_rows[["dataset", "experiment", "location", "localized_cases",
            "location_TP", "location_TN", "location_FP", "location_FN", "location_f1",
            "region_exact_rate", "region_jaccard", "region_presence_TP", "region_presence_TN",
            "region_presence_FP", "region_presence_FN", "region_presence_f1",
            "side_agreement_rate", "location_excluded"]].set_index(["dataset", "experiment"]))

    # Occupied-region counts: the number of distinct cells a finding is reported in against the number of
    # distinct cells its true boxes occupy (one cell per box). Scored only when both sides are resolved;
    # exact_rate_of_expected charges every unresolved, partial or unlocated prediction against the run.
    count_rows = LEADERBOARD[(LEADERBOARD["counting"] == True) & (LEADERBOARD["location"] != "none")]
    if not count_rows.empty:
        print("===== occupied-region counts (distinct regions per finding, never teeth) =====")
        display(count_rows[["dataset", "experiment", "location", "count_expected_count_checks",
            "count_scored_count_checks", "count_excluded_count_checks", "count_exact_rate", "count_mae",
            "count_overcount_rate", "count_undercount_rate",
            "count_exact_rate_of_expected"]].set_index(["dataset", "experiment"]))

if not FINDING_COMPARISON.empty:
    assessed = FINDING_COMPARISON[FINDING_COMPARISON["assessment_status"] != "not_assessed"]
    print("===== each assessed finding =====")
    display(assessed[["dataset", "experiment", "condition", "trained_task", "annotated_images",
        "annotated_positives", "scored_checks", "TP", "TN", "FP", "FN", "unparseable",
        "coverage", "sensitivity", "specificity", "ppv", "f1"]].set_index(
        ["dataset", "experiment", "condition"]))
    unassessed = FINDING_COMPARISON[FINDING_COMPARISON["assessment_status"] == "not_assessed"]
    if not unassessed.empty:
        print("===== annotated findings not assessed by this DentVLM protocol =====")
        display(unassessed[["dataset", "experiment", "condition", "trained_task",
            "annotated_images", "annotated_positives", "not_assessed_checks"]].set_index(
            ["dataset", "experiment", "condition"]))

if not SITUATION_COMPARISON.empty:
    print("===== case situations (descriptive slices) =====")
    display(SITUATION_COMPARISON[["dataset", "experiment", "situation", "group", "images_scored",
        "not_assessed_checks", "expected_finding_checks", "scored_finding_checks", "TP", "TN",
        "FP", "FN", "coverage", "f1", "regions_scored",
        "regions_exact_set_match_rate"]].set_index(["dataset", "experiment", "situation", "group"]))

# Native DentVLM sub-experiments are summarized here; condition/task rows and image IDs are in overview/*.csv.
if not STAGE_COMPARISON.empty:
    stage_all = STAGE_COMPARISON[STAGE_COMPARISON["condition"] == "ALL"]
    print("===== saved stage transitions =====")
    display(stage_all[["dataset", "experiment", "comparison", "transition", "checks"]].set_index(
        ["dataset", "experiment", "comparison", "transition"]))
if not PHRASING_COMPARISON.empty:
    print("===== phrasing agreement =====")
    phrasing_summary = PHRASING_COMPARISON.groupby(["dataset", "experiment", "status"], as_index=False)[
        ["checks", "unresolved_phrasings"]].sum()
    display(phrasing_summary.set_index(["dataset", "experiment", "status"]))
if not VOTE_REPLAY_COMPARISON.empty:
    print("===== union vs majority replay of saved rationale regions =====")
    display(VOTE_REPLAY_COMPARISON[["dataset", "experiment", "region_vote", "TP", "TN", "FP",
        "FN", "coverage", "f1", "regions_scored", "regions_excluded",
        "regions_exact_set_match_rate", "regions_mean_jaccard"]].set_index(
        ["dataset", "experiment", "region_vote"]))
if not PARSE_RECOVERY_COMPARISON.empty:
    print("===== parse outcomes =====")
    recovery_summary = PARSE_RECOVERY_COMPARISON.groupby(
        ["dataset", "experiment", "stage", "status"], as_index=False)["checks"].sum()
    display(recovery_summary.set_index(["dataset", "experiment", "stage", "status"]))
if not CALL_USAGE_COMPARISON.empty:
    print("===== analyzer usage =====")
    display(CALL_USAGE_COMPARISON[["dataset", "experiment", "stage", "attempt", "calls",
        "prompt_tokens", "completion_tokens", "latency_seconds"]].set_index(
        ["dataset", "experiment", "stage", "attempt"]))
# How the replies were read, per parser stage: how often the strict reader answered, how often it
# failed and the parser model was called, how often that model answered, and what stayed unresolved.
# Empty (and hidden) for a run read with code alone.
if not PARSER_USAGE_COMPARISON.empty:
    print("===== how model text was read (parser stages) =====")
    display(PARSER_USAGE_COMPARISON[["dataset", "experiment", "stage", "parses", "code_ok",
        "code_failed", "fallbacks", "llm_calls", "llm_ok", "cache_hits", "retries",
        "unresolved"]].set_index(["dataset", "experiment", "stage"]))

# Paired comparison against the first complete experiment: the same images and the same ground truth, so the
# columns say what a knob changed (checks corrected/worsened), not what the image sample was. Location and
# counts are left out here because each experiment has its own location truth in the tables above.
COMPARISONS = {}
for spec in DATASETS:
    dataset = spec["name"]
    complete = {c["name"]: xp.run_dir(c, dataset) for c in EXPERIMENTS
                if (c["name"], dataset) in REPORTS and not REPORTS[c["name"], dataset]["missing_results"]}
    if len(complete) < 2:
        continue
    # compare_runs re-checks that the runs answered byte-identical images; a mismatch is reported here
    # rather than quietly compared.
    with mon.guard(f"compare {dataset}", LEDGER) as step:
        COMPARISONS[dataset] = da.compare_runs(GT[dataset], complete, dataset=dataset, evaluate_location=False,
                                               counting=False)
        ev.write_report(COMPARISONS[dataset], Path(OUTPUT_ROOT) / "comparison" / dataset)
        print(f"\n===== {dataset}: paired against {next(iter(complete))} =====")
        display(pd.DataFrame(COMPARISONS[dataset]["run_comparison"])[
            ["run", "model", "phrasings", "region_vote", "location", "ask_untrained",
             "not_assessed_checks", "expected_finding_checks", "scored_finding_checks", "TP", "TN", "FP",
             "FN", "coverage", "sensitivity", "specificity", "ppv", "f1", "paired_checks",
             "paired_reference_f1", "paired_run_f1", "paired_f1_delta", "corrected", "worsened",
             "left_not_assessed", "became_not_assessed", "left_unresolved", "became_unresolved",
             "mean_calls_per_image", "mean_inference_calls_per_image", "cache_hits", "cache_hit_rate",
             "completion_tokens", "inference_completion_tokens"]].set_index("run"))
print(f"\nFull joined tables: {Path(OUTPUT_ROOT) / 'overview'}")
print("Coverage = scored / expected checks. Not-assessed findings have no active DentVLM task and are never "
      "counted as negatives. Unparseable answers are excluded from TP/FP/TN/FN and reported separately. "
      "Empty denominators stay N/A. Paired F1 uses only checks asked and resolved by both runs. A count is the "
      "number of distinct regions, never of teeth, and is scored only when the prediction and the truth are both resolved.")

# What did not finish, and why: every failure of every cell above, with the full
# tracebacks saved next to the runs.
LEDGER.report(path=Path(OUTPUT_ROOT) / "failures.json")

In [ ]:
# ============================================================
# CELL 12 - Drill into one experiment without losing its DentVLM-specific diagnostics
# ============================================================
INSPECT_EXPERIMENT = EXPERIMENTS[0]["name"]  # any name from CELL 3
INSPECT_DATASET = DATASETS[0]["name"]

report = REPORTS.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if report is None:
    print(f"No evaluation for {INSPECT_EXPERIMENT}/{INSPECT_DATASET}; run CELLs 9 and 11 first.")
else:
    print(f"===== {INSPECT_EXPERIMENT} / {INSPECT_DATASET} =====")
    selected_overview = LEADERBOARD[(LEADERBOARD["experiment"] == INSPECT_EXPERIMENT) &
                                    (LEADERBOARD["dataset"] == INSPECT_DATASET)]
    display(selected_overview.set_index(["dataset", "experiment"]))
    selected_findings = FINDING_COMPARISON[(FINDING_COMPARISON["experiment"] == INSPECT_EXPERIMENT) &
                                           (FINDING_COMPARISON["dataset"] == INSPECT_DATASET)]
    if not selected_findings.empty:
        display(selected_findings[["condition", "assessment_status", "trained_task", "annotated_images",
            "annotated_positives", "not_assessed_checks", "scored_checks", "TP", "TN", "FP", "FN",
            "unparseable", "coverage", "sensitivity", "specificity", "ppv", "f1", "whole_image_f1",
            "localized_cases", "location_f1",
            "region_exact_rate", "region_jaccard", "count_scored_count_checks", "count_excluded_count_checks",
            "count_exact_rate", "count_mae"]].set_index("condition"))
    if report["whole_image"]:
        print("whole-image answers alone (what the cells recovered, and what it cost):")
        display(pd.DataFrame(report["whole_image"]).set_index("condition"))
    if report["regions"]:
        display(pd.DataFrame(report["regions"]).set_index("condition"))
    if report["region_presence"]:
        print("presence per cell (every cell of every image, whatever the whole image said; rationale: named = present):")
        display(pd.DataFrame(report["region_presence"]).set_index(["condition", "region"]))
    if report.get("occupied_regions"):
        print("occupied-region counts (distinct regions a finding is reported in vs. the regions its true boxes occupy; "
              "'whole_image' rows are the rationale stage alone in the region comparison; excluded cases by reason):")
        display(pd.DataFrame(report["occupied_regions"]).set_index(["condition", "stage"]))
    # Full columns and per-finding transitions remain in JSON/CSV, with supporting image IDs.
    for table, title, columns in [
        ("stage_changes", "Saved decisions: first phrasing -> vote; whole image -> region questions",
         ["comparison", "transition", "checks"]),
        ("phrasing_votes", "Agreement of saved phrasings (after any parse repairs)",
         ["task", "status", "checks", "unresolved_phrasings"]),
        ("region_vote_comparison", "Union vs majority replay on the same saved rationale answers",
         ["region_vote", "expected_finding_checks", "coverage", "regions_scored", "regions_excluded",
          "regions_exact_set_match_rate", "regions_mean_jaccard"]),
        ("parse_recovery", "Recorded question recovery (composite/extra tasks lack individual correctness labels)",
         ["stage", "task", "status", "checks", "correctness_scored", "correct", "correct_rate"]),
        ("parser_usage", "How this run's replies were read (strict reader, parser model, unresolved)",
         ["stage", "parses", "code_ok", "code_failed", "fallbacks", "llm_calls", "llm_ok",
          "cache_hits", "retries", "unresolved"]),
        ("call_usage", "Recorded analyzer completions and parse-retry usage",
         ["stage", "attempt", "calls", "prompt_tokens", "prompt_tokens_recorded_calls",
          "completion_tokens", "completion_tokens_recorded_calls", "latency_seconds", "latency_seconds_recorded_calls"]),
        ("case_breakdown", "Case breakdowns (named cells and region counts are location evidence, not tooth counts)",
         ["situation", "group", "not_assessed_checks", "expected_finding_checks", "coverage", "TP", "TN", "FP", "FN",
          "sensitivity", "specificity", "regions_scored", "regions_exact_set_match_rate"]),
    ]:
        rows = report.get(table, [])
        if table == "stage_changes":
            rows = [r for r in rows if r["condition"] == "ALL"]
        if rows:
            print(title)
            display(pd.DataFrame(rows)[columns])
    selected_situation_findings = SITUATION_FINDING_COMPARISON[
        (SITUATION_FINDING_COMPARISON["experiment"] == INSPECT_EXPERIMENT) &
        (SITUATION_FINDING_COMPARISON["dataset"] == INSPECT_DATASET)]
    if not selected_situation_findings.empty:
        print("Situation x finding detail (full rows and image IDs are saved in overview/):")
        display(selected_situation_findings[["situation", "group", "condition", "assessment_status",
            "annotated_images", "not_assessed_checks", "TP", "TN", "FP", "FN", "coverage", "f1",
            "localized_cases", "location_f1",
            "region_exact_rate"]].set_index(["situation", "group", "condition"]))
    if not report["parse_recovery"]:
        print("No saved question-attempt metadata for recovery analysis.")
    datasets = [r for (n, d), r in REPORTS.items() if n == INSPECT_EXPERIMENT]
    if len(datasets) > 1:
        print("\n===== pooled over the findings every dataset scores =====")
        pooled = ev.pooled_presence(datasets)
        if pooled:
            display(pd.DataFrame(pooled).set_index("condition"))
print("Case slices are descriptive. Empty denominators are N/A; coverage excludes not-assessed findings. "
      "Location scores use each run's true positives. Replay uses saved, repaired answers, not a retries-OFF run.")

In [ ]:
# ============================================================
# CELL 13 - Side convention check, then one image: dentist summary and raw answers
# ============================================================
# DentVLM's "left"/"right" follow Table S6 (its "left posterior" = FDI quadrants 1 and 4, the patient's right,
# which is the image left). Agreement far above 50% confirms dental_pipeline.LEFT_IS_IMAGE_LEFT. Far below means
# set LEFT_IS_IMAGE_LEFT = False in dental_pipeline.py, run `import importlib; importlib.reload(dp); importlib.reload(ev)`,
# and re-run CELL 11: no new model calls, the saved replies keep DentVLM's own words and the adapted truth is re-mapped.
INSPECT_IMAGE = None   # None = first image of the dataset
SHOW_RAW_CALLS = False

cfg = next(c for c in EXPERIMENTS if c["name"] == INSPECT_EXPERIMENT)
results = dp.load_results(xp.run_dir(cfg, INSPECT_DATASET))
if cfg["evaluate_location"]:
    print(INSPECT_DATASET, "side agreement:", ev.side_agreement(GT[INSPECT_DATASET], results))

image_id = INSPECT_IMAGE or next(iter(sorted(results)))
result = results[image_id]
print(f"\n{INSPECT_EXPERIMENT} / {INSPECT_DATASET} / {image_id}\n")
print(dp.dentist_report(result, counting=cfg["counting"]))
truth = [b["condition"] for b in GT[INSPECT_DATASET][image_id]["boxes"]]
print("\nGround-truth boxes:", {c: truth.count(c) for c in dict.fromkeys(truth)} or "none")
adapted = ADAPTED.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if adapted:
    for k, record in enumerate(adapted[image_id]["boxes"], 1):
        print(f"  box {k}: {record['condition']} -> {record['regions']} ({record['source']}; fixed windows {record['geometry']})")
if cfg["counting"] and truth:
    # The count target: one region per true box (the adapted, FDI or fixed-window cell; a box that several
    # cells hold goes to the one holding most of it), deduplicated per finding.
    entry = (ev.apply_adapted({image_id: GT[INSPECT_DATASET][image_id]}, adapted) if adapted else GT[INSPECT_DATASET])[image_id]
    for condition in dict.fromkeys(truth):
        boxes = [b for b in entry["boxes"] if b["condition"] == condition]
        placed = sorted({ev.box_primary_region(b) for b in boxes} - {None})
        print(f"  {condition}: {len(boxes)} box(es) occupy {ev.truth_count(boxes)} region(s) {placed}"
              f" | predicted: {result['findings'][condition]['region_count']} ({result['findings'][condition]['count_status']})")
if SHOW_RAW_CALLS:
    for call in result["calls"]:
        print(f"\n--- {call['stage']} | {call['task']} | {call['cell']} | finish={call['finish_reason']}")
        print(call["text"][:800])

In [ ]:
# ============================================================
# CELL 14 - Dentist report: one LLM call per image over the structured findings (resumable)
# ============================================================
# DentVLM answered 13 yes/no questions per image (39 with three phrasings, 91 with region questions) and
# named locations in its rationales. The report writer gets them as one dense JSON (every benchmark finding and extra
# task with its verbatim question and parsed answer, every region, the multiplicity (the number of regions the
# finding was reported in, never teeth, with the "counting" knob; left out when it is off), explicit statuses incl.
# not_assessed), returns a report in the experiment's report_language as JSON (one entry per finding in seven
# sections, impression, caveats), which is verified against the data (every finding exactly once, statuses unchanged,
# nothing invented), sent back once for correction if it fails, and rendered to Markdown. One .json + one .md per
# image under <name>/<dataset>/reports/<model>-<language>/reports; a reply that fails twice keeps the deterministic
# dentist summary, marked as such. The model never sees the image.
# With the reporter's "vote_agreement" knob on (CELL 3) and phrasings > 1, each finding also carries the vote behind
# it - how many readable answers reported it, how many wordings were asked, how many were unreadable, whether they
# tied, and how many of the reporting answers named each region separately - and the report describes that agreement
# in calibrated words ("consistently identified", "moderately supported", "weakly supported"). Those counts say how
# stable the model is under rewording; they are not a probability, medical certainty or diagnostic confidence, and
# the report says so. The answers, the union/majority decision and the evaluation are unchanged.
from IPython.display import Markdown, display

REPORT_EXPERIMENTS = [INSPECT_EXPERIMENT]  # one call per image per experiment; [c["name"] for c in EXPERIMENTS] for all

WRITTEN = {}
for cfg in [c for c in EXPERIMENTS if c["name"] in REPORT_EXPERIMENTS]:
    name = cfg["name"]
    with mon.guard(f"report writer {name}", LEDGER) as step:
        writer = xp.report_writer(cfg, parser=open_parser(cfg))
        print(f"{name}: report writer {writer.public()}")
    if not step.ok:
        continue
    for spec in DATASETS:
        dataset = spec["name"]
        with mon.guard(f"reports {name}/{dataset}", LEDGER) as step:
            results = dp.load_results(xp.run_dir(cfg, dataset))
            if not results:
                LEDGER.note(f"reports {name}/{dataset}", "no results to report; run CELL 9")
                continue
            WRITTEN[name, dataset] = rw.report_dataset(
                writer, results, xp.run_dir(cfg, dataset) / "reports" / writer.run_name,
                analyzer=xp.analyzer_name(cfg), resume=True, limit=cfg["report_images"], ledger=LEDGER)
            print(f"{name}/{dataset}: {rw.summarize_reports(WRITTEN[name, dataset])}")

# One report to read (the image inspected in CELL 13 when it has one).
reports = WRITTEN.get((INSPECT_EXPERIMENT, INSPECT_DATASET)) or next(iter(WRITTEN.values()), {})
if reports:
    shown = reports.get(globals().get("image_id")) or reports[next(iter(sorted(reports)))]
    print(f"{shown['image_id']}: verified={shown['verified']} | attempts={len(shown['attempts'])} | "
          f"problems={shown['problems']}")
    display(Markdown(shown["markdown"]))